In [1]:
import numpy as np
import wfdb
import matplotlib.pyplot as plt
import pywt
from sklearn.preprocessing import RobustScaler

import cv2
from scipy import signal

from sklearn.preprocessing import OneHotEncoder

import torch.optim as optim
import os
import gc # Сборщик мусора

In [2]:
import torch
from torch.utils.data import Dataset
import os
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

import torch.nn as nn
import torch.nn.functional as F


# Data assembling

In [3]:
def get_dwt_scalogram(window, wavelet='db4', level=5, shape=(125, 120)):

    h, w = shape
    ecg_data = window.T
    all_scalograms = []
    
    for lead_signal in ecg_data:

        coeffs = pywt.wavedec(lead_signal, wavelet, level=level)
        stack = [cv2.resize(np.abs(c).reshape(1, -1), (w, 1))[0] for c in coeffs[::-1]]
        
        scalogram = np.vstack(stack)
        if scalogram.shape[0] != h:
            scalogram = cv2.resize(scalogram, (w, h), interpolation=cv2.INTER_LINEAR)

        all_scalograms.append(np.log1p(scalogram))
    
    multi_channel_input = np.stack(all_scalograms, axis=-1)
        
    return multi_channel_input

def prepare_stft_data(window, fs=500, nperseg=256, noverlap=128):

    ecg_data = window.T
    all_spectrograms = []
    
    for lead_signal in ecg_data:
        
        f, t, Zxx = signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
        amplitude = np.abs(Zxx)
        
        spec_log = np.log10(amplitude + 1e-6)
        spec_norm = (spec_log - np.mean(spec_log)) / (np.std(spec_log) + 1e-8)
        
        all_spectrograms.append(spec_norm)
    
    multi_channel_input = np.stack(all_spectrograms, axis=-1)
    
    return multi_channel_input

# X_dwt = get_dwt_scalogram(cut_data[0])
# print(X_dwt.shape)

In [4]:
with open ("path_full.txt", 'r', encoding='utf-8') as f:
    path_full = f.read()
g_list = ["g1\\","g2\\","g3\\","g4\\"]

In [5]:
RECORDS = []
for g in g_list:
    with open (path_full+g+"RECORDS", 'r', encoding='utf-8') as f:
        RECORDS += f.read().split("\n")[:-1]

In [6]:
with open('train_names.txt', 'r', encoding='utf-8') as f:
    train_names = f.read().splitlines()
with open('val_names.txt', 'r', encoding='utf-8') as f:
    val_names = f.read().splitlines()

In [7]:
def onehot_group(target):
    oh = np.zeros(12)
    st_t_codes = {'428750005', '164930006', '164934002', '429622005'} # Изменения ST/T
    rare_imp = {'111975006', '164909002', '195042002', '54329005'} # QT, БЛНПГ, и т.д.

    for code in target:
        # 0. Норма
        if code == '427084000' or code == '164865005': 
            oh[0] = 1
        # 1. Изменения ST-T
        elif code in st_t_codes:
            oh[1] = 1
        # 2. Мерцалка/Трепетание (AF/AFL)
        elif code == '164889003' or code == '164890007':
            oh[2] = 1
        # 3. Брадикардия
        elif code == '164861001':
            oh[3] = 1
        # 4. Желудочковая экстрасистолия (PVC)
        elif code == '427172004':
            oh[4] = 1
        # 5. Предсердная экстрасистолия (PAC)
        elif code == '284470004':
            oh[5] = 1
        # 6. Блокада левой ножки (LBBB)
        elif code == '164909002':
            oh[6] = 1
        # 7. Блокада правой ножки (RBBB)
        elif code == '164867002' or code == '164931005':
            oh[7] = 1
        # 8. Отклонение оси влево (LAD)
        elif code == '164873001':
            oh[8] = 1
        # 9. АВ-блокада 1 степени
        elif code == '270492004':
            oh[9] = 1
        # 10. Тахикардии
        elif code == '426177001' or code == '426761007':
            oh[10] = 1
        # 11. Rare Important (Опасные/Важные редкие)
        elif code in rare_imp:
            oh[11] = 1
            
    return oh

In [8]:
all_code_freqs_gr = np.zeros(12)
for g in g_list:
    for rec_id in RECORDS:
        try:
            rec = wfdb.rdrecord(path_full+g+rec_id)
        except FileNotFoundError:
            # print(rec_ID)
            continue
        # data_rec = rec.p_signal
        header_rec = rec.comments
        diag = header_rec[2][4:].split(",")
    
        all_code_freqs_gr += onehot_group(diag)
all_code_freqs_gr = all_code_freqs_gr.astype(int)
all_code_freqs_gr

array([ 660, 1848,  207,  384,  188,   73,   38, 1227,  158,  106,   48,
         87])

In [9]:
super_rare = [5,6,10,11]

In [10]:
num_classes = 12
batch_size_records = 150 # Сохраняем каждые 150 пациентов
window_size = 2500
slide = 500

In [ ]:
X_dwt_full, X_stft_full, y, rec_ids_full = [], [], [], []

os.makedirs('E:/processed_data', exist_ok=True)
os.makedirs('E:/processed_data/train', exist_ok=True)
os.makedirs('E:/processed_data/val', exist_ok=True)

In [11]:
def normalize_signal(signal):
    """
    Приводим сигнал к среднему 0 и отклонению 1.
    Это защитит нас от разницы в усилении АЦП.
    """
    mean = np.mean(signal)
    std = np.std(signal) + 1e-6
    return (signal - mean) / std

In [12]:
def window_slide(data, wind_len=2500, slide=500):
    starts = np.arange(0, max(1, len(data) - wind_len + slide), slide)
    all_data = []
    
    for s in starts:
        end = min(s + wind_len, len(data))
        start = s if s + wind_len <= len(data) else len(data) - wind_len
        all_data.append(data[start:start + wind_len])
        
    return np.array(all_data)[..., :8]

In [ ]:
batch_count = 0

for g in g_list:
    for i, rec_id in enumerate(RECORDS):
        try:
            rec = wfdb.rdrecord(path_full+g+rec_id)
        except FileNotFoundError:
            continue
        
        data_rec = rec.p_signal
        
        header_rec = rec.comments
        diag = header_rec[2][4:].split(",")
        target = onehot_group(diag)

        # Самые редкие нарезаем с нахлёстом
        if np.any([target[j] for j in super_rare]):
            cut_data = window_slide(data_rec, window_size, slide)
        else:
            cut_data = window_slide(data_rec, window_size, window_size)
        
        for window in cut_data:
            
            if np.any(target):# непустые векторы
                window_scaled = normalize_signal(window)
                X_dwt_full.append(get_dwt_scalogram(window_scaled))
                X_stft_full.append(prepare_stft_data(window_scaled))
                y.append(target)
                rec_ids_full.append(rec_id)
            else:
                continue
        
        # Если накопили достаточно или это последний файл
        if (i + 1) % batch_size_records == 0:
            sub_folder = 'train' if rec_id in train_names else 'val'
            # Превращаем в массивы и сохраняем
            np.save(f'E:/processed_data/{sub_folder}/X_dwt_{batch_count}.npy', np.array(X_dwt_full, dtype=np.float32))
            np.save(f'E:/processed_data/{sub_folder}/X_stft_{batch_count}.npy', np.array(X_stft_full, dtype=np.float32))
            np.save(f'E:/processed_data/{sub_folder}/y_{batch_count}.npy', np.array(y, dtype=np.float32))
            np.save(f'E:/processed_data/{sub_folder}/rec_ids_{batch_count}.npy', np.array(rec_ids_full))
            
            # ОЧИСТКА
            X_dwt_full, X_stft_full, y, rec_ids_full = [], [], [], []
            gc.collect() # Принудительно очищаем память
            batch_count += 1
            print(f"Batch {batch_count} saved.")


In [14]:
np.load(f'E:/processed_data/train/X_stft_{19}.npy').shape, np.load(f'E:/processed_data/train/X_dwt_{19}.npy').shape

((571, 129, 21, 8), (571, 125, 120, 8))

In [16]:
prefix_y='y_'
data_dir="E:/processed_data/train/"
y_files_train = sorted(["E:/processed_data/train/"+f for f in os.listdir("E:/processed_data/train") if f.startswith(prefix_y)])

y_freqs = np.zeros(12)
total_windows = 0

for y_path in y_files_train:
    y = np.load(y_path)
    y_freqs += np.sum(y, axis=0)
    total_windows += len(y)
y_freqs, total_windows

(array([1953., 6809.,  596.,  824.,  915.,  975.,  403., 3659.,  665.,
         331.,  417.,  669.]),
 11343)

In [17]:
weights = (total_windows - y_freqs) / (y_freqs + 1e-6)
weights = np.log1p(weights)

In [18]:
rare = [5,6,10,11]

paths_common = []
idx_common = []

paths_rare = []
idx_rare = []

for y_path in y_files_train:
    y = np.load(y_path)
    for i in range(len(y)):
        paths_common.append(y_path)
        idx_common.append(i)
        
        if np.any([y[i][j] for j in rare]):
            paths_rare.append(y_path)
            idx_rare.append(i)

In [19]:
class ECGDataset(Dataset):
    def __init__(self, paths_for_ft, idx_for_ft, paths_rare, idx_rare, noise_level=0.05):
        self.paths = paths_for_ft + paths_rare
        self.indices = idx_for_ft + idx_rare
        self.is_rare = [False] * len(paths_for_ft) + [True] * len(paths_rare)
        self.noise_level = noise_level

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        y_path = self.paths[i]
        local_idx = self.indices[i]
        
        folder, filename = os.path.split(y_path)
        # Путь к файлу с ID записей
        id_path = os.path.join(folder, filename.replace('y_', 'rec_ids_'))
        dwt_path = os.path.join(folder, filename.replace('y_', 'X_dwt_'))
        stft_path = os.path.join(folder, filename.replace('y_', 'X_stft_'))

        try:
            # Загружаем ID
            ids_mmap = np.load(id_path, mmap_mode='r')
            rec_id = str(ids_mmap[local_idx])
            # Открываем mmap прямо внутри getitem. 
    
            dwt_mmap = np.load(dwt_path, mmap_mode='r')
            stft_mmap = np.load(stft_path, mmap_mode='r')
            y_mmap = np.load(y_path, mmap_mode='r')

            # Сразу забираем данные и "отклеиваемся" от файла через np.array()
            x_dwt = np.array(dwt_mmap[local_idx], dtype=np.float32)
            x_stft = np.array(stft_mmap[local_idx], dtype=np.float32)
            target = np.array(y_mmap[local_idx], dtype=np.float32)
            
            # Деструкторы закроют mmap при выходе из функции
        except Exception as e:
            print(f"Error loading {y_path}: {e}")
            # Возвращаем нулевые тензоры в случае ошибки, чтобы не падал цикл
            return torch.zeros((8, 125, 120)), torch.zeros((8, 129, 21)), torch.zeros(12), ""

        if self.is_rare[i] and self.noise_level > 0:
            x_dwt += np.random.normal(0, self.noise_level, x_dwt.shape).astype(np.float32)
            x_stft += np.random.normal(0, self.noise_level, x_stft.shape).astype(np.float32)

        x_dwt = torch.from_numpy(x_dwt).permute(2, 0, 1)
        x_stft = torch.from_numpy(x_stft).permute(2, 0, 1)
        target = torch.from_numpy(target)

        return x_dwt, x_stft, target, rec_id

In [20]:
# Создаем датасет
train_dataset = ECGDataset(
    paths_for_ft=paths_common, 
    idx_for_ft=idx_common, 
    paths_rare=paths_rare, 
    idx_rare=idx_rare,
    noise_level=0.05
)

# Важно: shuffle=True обязателен, т.к. в Dataset данные идут блоками (сначала обычные, потом редкие)
train_loader = DataLoader(
    train_dataset, 
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True, 
)

In [21]:
val_data_dir = "E:/processed_data/val/"
prefix_y = 'y_'

# Получаем список всех y_ файлов в валидационной папке
y_files_val = sorted([os.path.join(val_data_dir, f) for f in os.listdir(val_data_dir) if f.startswith(prefix_y)])

paths_val = []
idx_val = []

for y_path in y_files_val:
    y_data = np.load(y_path)
    for i in range(len(y_data)):
        paths_val.append(y_path)
        idx_val.append(i)

print(f"Валидация: найдено {len(paths_val)} окон.")

Валидация: найдено 2528 окон.


In [22]:
val_dataset = ECGDataset(
    paths_for_ft=paths_val,   # Все валидационные окна
    idx_for_ft=idx_val,
    paths_rare=[],            # На валидации не дополняем
    idx_rare=[],
    noise_level=0             # На валидации не зашумляем
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=64, 
    shuffle=False,      # Для валидации порядок не важен, ставим False
    num_workers=0
)

# NN

In [23]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class SEAttention(nn.Module):
    """ Простой механизм внимания для взвешивания каналов """
    def __init__(self, channels, reduction=4):
        super(SEAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ECGMultiModalNet(nn.Module):
    def __init__(self, num_classes):
        super(ECGMultiModalNet, self).__init__()
        self.dwt_branch = nn.Sequential( # Ветка DWT
            nn.Conv2d(8, 32, kernel_size=(3, 7), padding=(1, 3)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.stft_branch = nn.Sequential( # Ветка STFT
            nn.Conv2d(8, 32, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.classifier = nn.Sequential( # Общий классификатор
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x_dwt, x_stft):
        feat_dwt = self.dwt_branch(x_dwt)
        feat_stft = self.stft_branch(x_stft)
        combined = torch.cat((feat_dwt, feat_stft), dim=1)
        return self.classifier(combined)

In [24]:
def train_model(model, train_loader, val_loader, device, epochs=10, threshold=0.3, start_epoch=0, path='last_checkpoint', best_val_f1 = 0.0):

    patience_counter = 0
    patience_limit = 5
    
    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        
        all_preds = []
        all_labels = []
        
        for x_dwt, x_stft, labels, _ in train_loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(x_dwt, x_stft)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Переводим логиты в вероятности и применяем порог
            preds = (torch.sigmoid(outputs) > threshold).int()
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
        # Склеиваем всё в один массив для расчета метрик
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        
        # Считаем F1 (macro — среднее по каждому из 72 классов)
        train_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        
        # Валидация
        val_loss, val_f1 = validate(model, val_loader, criterion, device, threshold)
        
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_f1': best_val_f1
        }
        torch.save(checkpoint, path+'.pth')

        # Если модель стала лучше — сохраняем её как "лучшую"
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(checkpoint, f"{path}_best.pth")
            patience_counter = 0
            print(f"New best Val F1: {val_f1:.4f}. Model saved!")
        else:
            patience_counter += 1
            
        if patience_counter > patience_limit:
            print("Early stopping triggered.")
            break
        
        print(f"Epoch {epoch+1} | LR: {current_lr:.6f}")
        print(f"Train Loss: {running_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        print("-" * 30)

def validate(model, loader, criterion, device, threshold=0.3):
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x_dwt, x_stft, labels, _ in loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            outputs = model(x_dwt, x_stft)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Применяем сигмоиду и новый порог
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).int()
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Считаем F1 (Macro усредняет по всем 12 классам независимо от их частоты)
    val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return val_loss/len(loader), val_f1

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем: {device}")

model = ECGMultiModalNet(num_classes=num_classes).to(device)

pos_weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.6, patience=3, threshold=0.001, threshold_mode='abs')

In [ ]:
train_model(model, train_loader, val_loader, device, epochs=15, path="last_checkpoint_0505_2")

# Restart exaxmple

In [25]:
def load_checkpoint(checkpoint_path, model, optimizer, scheduler):
    if os.path.exists(checkpoint_path):
        print(f"==> Загружаем чекпоинт: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        
        # Восстанавливаем всё содержимое
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        # раньше не сохранял best_val_f1, поэтому он не во всех чекпоинтах
        try:
            best_val_f1 = checkpoint['best_f1']
        except KeyError:
            best_val_f1 = 0
                
        print(f"Успешно! Начинаем с эпохи {start_epoch}")
        return start_epoch, best_val_f1
    else:
        print("==> Чекпоинт не найден. Начинаем обучение с нуля.")
        return 0, 0.0

In [26]:
# if __name__ == '__main__':
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем: {device}")

model = ECGMultiModalNet(num_classes=num_classes).to(device)

pos_weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.6, patience=3, threshold=0.001, threshold_mode='abs')

checkpoint_path = 'test_checkpoint_1.pth'

# Загружаем прогресс
start_epoch, best_val_f1 = load_checkpoint(checkpoint_path, model, optimizer, scheduler)
for param_group in optimizer.param_groups:
    param_group['lr'] /=2 # Ставим в 10 раз меньше



Используем: cuda
==> Загружаем чекпоинт: test_checkpoint_1.pth
Успешно! Начинаем с эпохи 23


C:\Users\HundeRob0t\AppData\Local\Temp\ipykernel_16860\1854443200.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


In [27]:
# Передаем start_epoch в функцию обучения
train_model(
    model,
    train_loader, 
    val_loader, 
    device, 
    epochs=25, 
    start_epoch=start_epoch,
    path='test_checkpoint_1_new',
    best_val_f1=best_val_f1
)

New best Val F1: 0.3733. Model saved!
Epoch 24 | LR: 0.000054
Train Loss: 0.2041 | Train F1: 0.7538
Val Loss: 0.6827 | Val F1: 0.3733
------------------------------
New best Val F1: 0.3841. Model saved!
Epoch 25 | LR: 0.000032
Train Loss: 0.1834 | Train F1: 0.7592
Val Loss: 0.6595 | Val F1: 0.3841
------------------------------


# Validation & thresholds

In [28]:
def validate_model_with_aggregator(model, val_loader, device, thresholds=0.3):
    """
    Валидация с использованием Max-Pooling агрегатора по идентификаторам записей (rec_ids).
    thresholds: число или список/массив из 12 значений.
    """
    model.eval()
    model = model.to(device)
    
    all_agg_preds = []
    all_agg_labels = []
    total_loss = 0.0
    criterion = torch.nn.BCEWithLogitsLoss()

    # Подготовка порогов
    if isinstance(thresholds, (int, float)):
        thresholds = [thresholds] * 12
    thresholds = np.array(thresholds)

    # Буферы для агрегации текущего пациента
    temp_probs = []
    temp_labels = []
    last_rec_id = None

    with torch.no_grad():
        for x_dwt, x_stft, labels, rec_ids in val_loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            logits = model(x_dwt, x_stft)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits).cpu().numpy()
            labels_np = labels.cpu().numpy()

            # Идем по батчу и проверяем смену rec_id
            for i in range(len(probs)):
                curr_id = rec_ids[i]

                # Если ID сменился — пора агрегировать накопленное по предыдущему пациенту
                if last_rec_id is not None and curr_id != last_rec_id:
                    # Схлопываем вероятности и метки
                    agg_prob = np.max(temp_probs, axis=0)
                    agg_label = np.max(temp_labels, axis=0)
                    
                    # Применяем пороги
                    agg_pred = (agg_prob > thresholds).astype(float)
                    
                    all_agg_preds.append(agg_pred)
                    all_agg_labels.append(agg_label)
                    
                    # Очищаем буферы для нового пациента
                    temp_probs = []
                    temp_labels = []

                temp_probs.append(probs[i])
                temp_labels.append(labels_np[i])
                last_rec_id = curr_id

    # Не забываем обработать последнего пациента после выхода из цикла
    if temp_probs:
        agg_prob = np.max(temp_probs, axis=0)
        agg_label = np.max(temp_labels, axis=0)
        agg_pred = (agg_prob > thresholds).astype(float)
        
        all_agg_preds.append(agg_pred)
        all_agg_labels.append(agg_label)

    # Превращаем в финальные массивы
    all_agg_preds = np.vstack(all_agg_preds)
    all_agg_labels = np.vstack(all_agg_labels)
    
    metrics = {
        'val_loss': total_loss / len(val_loader),
        'f1_macro': f1_score(all_agg_labels, all_agg_preds, average='macro', zero_division=0),
        'f1_micro': f1_score(all_agg_labels, all_agg_preds, average='micro', zero_division=0),
        'precision': precision_score(all_agg_labels, all_agg_preds, average='macro', zero_division=0),
        'recall': recall_score(all_agg_labels, all_agg_preds, average='macro', zero_division=0)
    }

    print(f"\n--- Smart Aggregated Validation Results (By Patient ID) ---")
    print(f"Loss: {metrics['val_loss']:.4f}")
    print(f"F1 Macro: {metrics['f1_macro']:.4f} | F1 Micro: {metrics['f1_micro']:.4f}")
    print(f"Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}")
    print("\nDetailed Report:")
    print(classification_report(all_agg_labels, all_agg_preds, zero_division=0))
    
    return metrics

In [29]:
def model_agg_raw(model, val_loader, device):
    model.eval()
    all_agg_probs = []
    all_agg_labels = []
    
    temp_probs = []
    temp_labels = []
    last_rec_id = None

    with torch.no_grad():
        for x_dwt, x_stft, labels, rec_ids in val_loader:
            x_dwt, x_stft = x_dwt.to(device), x_stft.to(device)
            logits = model(x_dwt, x_stft)
            probs = torch.sigmoid(logits).cpu().numpy()
            labels_np = labels.cpu().numpy()

            for i in range(len(probs)):
                curr_id = rec_ids[i]

                # Если это новый пациент (и это не самая первая итерация)
                if curr_id != last_rec_id and last_rec_id is not None:
                    # Схлопываем всё, что накопили по ПРЕДЫДУЩЕМУ пациенту
                    all_agg_probs.append(np.max(temp_probs, axis=0))
                    all_agg_labels.append(np.max(temp_labels, axis=0))
                    temp_probs, temp_labels = [], []

                temp_probs.append(probs[i])
                temp_labels.append(labels_np[i])
                last_rec_id = curr_id

    # Не забываем сохранить последнего пациента из батча
    if temp_probs:
        all_agg_probs.append(np.max(temp_probs, axis=0))
        all_agg_labels.append(np.max(temp_labels, axis=0))

    # Дальше — подбор порогов или расчет метрик на vstack(all_agg_probs)
    return np.vstack(all_agg_probs), np.vstack(all_agg_labels)

In [30]:
def find_optimal_thresholds(all_probs, all_labels, min_threshold=0.1, min_precision=0.15):
    best_thresholds = []
    print(f"Поиск адекватных порогов (min_thresh={min_threshold}, min_prec={min_precision})...")
    
    for i in range(all_labels.shape[1]):
        y_true = all_labels[:, i]
        y_prob = all_probs[:, i]
        
        best_f1 = -1
        best_thresh = 0.3 # дефолт
        
        # Ищем порог от min_threshold до 0.8
        for thresh in np.arange(min_threshold, 0.81, 0.01):
            y_pred = (y_prob > thresh).astype(int)
            
            p = precision_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            
            # Условие: F1 должен быть лучшим, НО точность не должна быть мусорной
            if f1 > best_f1 and p >= min_precision:
                best_f1 = f1
                best_thresh = thresh
        
        # Если при всех порогах Precision ниже плинтуса, берем тот, где хотя бы какой-то F1
        if best_f1 == -1:
             # Повторный поиск без ограничения по Precision, но с лимитом по порогу
             for thresh in np.arange(min_threshold, 0.51, 0.01):
                f1 = f1_score(y_true, (y_prob > thresh).astype(int), zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_thresh = thresh

        best_thresholds.append(round(best_thresh, 2))
        print(f"Класс {i:2}: Thresh = {best_thresh:.2f}, F1 = {best_f1:.4f}")
        
    return best_thresholds

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load('last_checkpoint_0505_2.pth')
loaded_model = ECGMultiModalNet(num_classes=12).to(device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])

C:\Users\HundeRob0t\AppData\Local\Temp\ipykernel_16860\4286204233.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('last_checkpoint_0505_2.pth')


<All keys matched successfully>

In [43]:
probs, labels = model_agg_raw(loaded_model, val_loader, device)

# 2. Ищем пороги (с ограничением Precision и F0.5)
opt_thresholds = find_optimal_thresholds(probs, labels, min_threshold=0.1, min_precision=0.2)

# 3. Проверяем результат
print("Найденные пороги:", opt_thresholds)

🔍 Поиск адекватных порогов (min_thresh=0.1, min_prec=0.2)...
Класс  0: Thresh = 0.59, F1 = 0.6555
Класс  1: Thresh = 0.12, F1 = 0.7927
Класс  2: Thresh = 0.72, F1 = 0.6750
Класс  3: Thresh = 0.54, F1 = 0.3625
Класс  4: Thresh = 0.36, F1 = 0.1667
Класс  5: Thresh = 0.45, F1 = 0.1176
Класс  6: Thresh = 0.17, F1 = 0.2353
Класс  7: Thresh = 0.43, F1 = 0.6766
Класс  8: Thresh = 0.38, F1 = 0.3175
Класс  9: Thresh = 0.77, F1 = 0.5000
Класс 10: Thresh = 0.33, F1 = 0.3333
Класс 11: Thresh = 0.33, F1 = 0.5000
Найденные пороги: [0.59, 0.12, 0.72, 0.54, 0.36, 0.45, 0.17, 0.43, 0.38, 0.77, 0.33, 0.33]


In [42]:
opt_thresholds

[0.59, 0.12, 0.72, 0.54, 0.36, 0.45, 0.17, 0.43, 0.38, 0.77, 0.33, 0.33]

In [44]:
# Теперь прогоняем финальный отчет с этими порогами
results = validate_model_with_aggregator(
    loaded_model, 
    val_loader, 
    device, 
    thresholds=opt_thresholds, # Используем найденные пороги
)


--- Smart Aggregated Validation Results (By Patient ID) ---
Loss: 0.3298
F1 Macro: 0.4444 | F1 Micro: 0.6431
Precision: 0.4782 | Recall: 0.5031

Detailed Report:
              precision    recall  f1-score   support

           0       0.60      0.73      0.66       107
           1       0.69      0.94      0.79       349
           2       0.68      0.68      0.68        40
           3       0.27      0.54      0.36        54
           4       0.11      0.35      0.17        26
           5       0.20      0.08      0.12        12
           6       0.17      0.40      0.24         5
           7       0.58      0.81      0.68       224
           8       0.24      0.45      0.32        22
           9       0.67      0.40      0.50        20
          10       1.00      0.20      0.33        15
          11       0.54      0.47      0.50        15

   micro avg       0.55      0.77      0.64       889
   macro avg       0.48      0.50      0.44       889
weighted avg       0.59  

### Outline of inference function

In [ ]:
import numpy as np

def predict_live_stream(current_buffer, thresholds, model_fn):
    """
    Математический пайплайн для обработки сигнала в реальном времени.
    Вызывается при каждом обновлении буфера на 500 отсчетов (1 секунда).
    
    current_buffer: матрица (8, N), где N >= 7000 (последние 14 секунд ЭКГ)
    thresholds: массив из 12 оптимальных порогов из find_optimal_thresholds_v2
    model_fn: функция, которая принимает (dwt, stft) и возвращает логиты модели
    """
    window_size = 2500
    stride = 500
    agg_size = 10  # Собираем вывод по 10 последним окнам
    
    # 1. Поканальная нормализация всего доступного буфера
    mean = np.mean(current_buffer, axis=1, keepdims=True)
    std = np.std(current_buffer, axis=1, keepdims=True) + 1e-8
    normalized_signal = (current_buffer - mean) / std
    
    # 2. Нарезка на скользящие окна внутри буфера
    num_samples = normalized_signal.shape[1]
    window_probs = []
    
    for start in range(0, num_samples - window_size + 1, stride):
        # Вырезаем текущее 5-секундное окно
        window = normalized_signal[:, start:start + window_size]
        window_t = window.T  # Переворот в (2500, 8) для препроцессинга
        
        # 3. Преобразования признаков (DWT и STFT)
        dwt_img = get_dwt_scalogram(window_t)   # На выходе тензор нужной формы
        stft_img = prepare_stft_data(window_t)  # На выходе тензор нужной формы
        
        # Подгоняем размерность под вход сети (добавляем батч-си shrink: Batch=1, Channels=8, H, W)
        dwt_input = np.expand_dims(np.transpose(dwt_img, (2, 0, 1)), axis=0)
        stft_input = np.expand_dims(np.transpose(stft_img, (2, 0, 1)), axis=0)
        
        # 4. Получение логитов и расчет вероятности через классическую Сигмоиду
        logits = model_fn(dwt_input, stft_input)
        probs = 1.0 / (1.0 + np.exp(-logits))
        
        window_probs.append(probs)
        
    if not window_probs:
        return np.zeros(12), np.zeros(12)
        
    # Превращаем в массив формы (Total_Windows, 12)
    window_probs = np.vstack(window_probs)
    
    # 5. Агрегация: берем срез из последних 10 окон
    recent_windows = window_probs[-agg_size:]
    aggregated_probs = np.max(recent_windows, axis=0)  # Max-Pooling по времени
    
    # 6. Финальный вердикт по калиброванным порогам
    final_diagnoses = (aggregated_probs > np.array(thresholds)).astype(int)
    
    return aggregated_probs, final_diagnoses